# 04 - Experimental Matrix Audit & Monitoring Dashboard

**Scientific & Operational Objectives:**
1. **Dynamic Database Discovery**: Directly queries  and  to detect dimensions, noise regimes, problem IDs, and model architectures without hardcoded lists.
2. **Two-Tier Pipeline Audit**:
   - **Tier 1 (Evolutionary Phase - SQLite)**: Audits LLaMEA evolutionary synthesis runs, iteration counts, convergence frequencies, and database integrity.
   - **Tier 2 (Evaluation Phase - IOH Logs)**: Audits post-evolution =10$ benchmark evaluations for LLM champions and classical baselines (, , ).
3. **Sample Imbalance & Gap Detection**: Quantifies run distribution imbalances and flags unexecuted experimental cells.
4. **Automated Report & Visual Export**:
   - Generates 
   - Renders 
5. **Actionable Task Dispatcher**: Emits ready-to-run commands for any missing experimental condition.


In [1]:
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from shared.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from shared.database import create_db_session_factory
from benchmarking.infra.storage import EvaluationConfigRepository, SQLiteSynthesisReadRepository
from benchmarking.application.audit_service import EvaluationAuditService
from benchmarking.domain.enums import BBOBFunction
from benchmarking.domain.services.resolvers import resolve_folder_solver_name, format_db_solver_name

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
from benchmarking.infra.io.trace_repository import IOHTraceReader
IOH_TRACES_DIR = RESULTS_DIR / "ioh_traces"
trace_repo = IOHTraceReader(IOH_TRACES_DIR)
config_repo = EvaluationConfigRepository()
service = EvaluationAuditService(sqlite_repo=sqlite_repo, trace_repo=trace_repo, config_repo=config_repo)

DB_PATH = DATA_DIR / 'db.sqlite3'
IOH_TRACES_DIR  = RESULTS_DIR / 'ioh_traces'
EVALUATIONS_DIR = IOH_TRACES_DIR
AUDIT_DIR       = RESULTS_DIR / 'audit'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases
# BBOBFunction directly available


In [2]:
loader = service.get_global_audit_matrix()
print("🔍 Dynamic DB & Model Discovery Summary:")
print(f"   • Dimensions ({len(loader.dims)}): {loader.dims}")
print(f"   • Noise Levels ({len(loader.noise_levels)}): {loader.noise_levels}")
print(f"   • Problem IDs ({len(loader.problem_ids)}): {loader.problem_ids}")
print(f"   • Discovered Solvers ({len(loader.all_solvers)}):")
for s in loader.all_solvers:
    print(f"       - {s}")


🔍 Dynamic DB & Model Discovery Summary:
   • Dimensions (4): [2, 3, 5, 10]
   • Noise Levels (2): [0.0, 0.05]
   • Problem IDs (5): [1, 8, 11, 15, 21]
   • Discovered Solvers (15):
       - Baseline / CMAES
       - Baseline / DE
       - Baseline / PSO
       - Qwen2.5-Coder-14B / baseline
       - Qwen2.5-Coder-14B / guided
       - Qwen2.5-Coder-14B / thinking
       - Qwen2.5-Coder-14B / vectorization
       - Qwen2.5-Coder-3B / baseline
       - Qwen2.5-Coder-3B / guided
       - Qwen2.5-Coder-3B / thinking
       - Qwen2.5-Coder-3B / vectorization
       - Qwen2.5-Coder-7B / baseline
       - Qwen2.5-Coder-7B / guided
       - Qwen2.5-Coder-7B / thinking
       - Qwen2.5-Coder-7B / vectorization


In [3]:
# ── 3. Compile Multi-Tier Experimental Audit DataFrames ───────────────────
target_runs = service.target_runs
eval_records = []
for d in loader.dims:
    for n in loader.noise_levels:
        for p in loader.problem_ids:
            for s in loader.all_solvers:
                cnt = loader.eval_counts[(d, n, p)][s]
                eval_records.append({
                    "Dimension": f"{d}D",
                    "Noise": f"σ={n}",
                    "Problem_ID": p,
                    "Problem_Name": BBOBFunction.get_name(p),
                    "Hardness_Class": BBOBFunction.get_class(p),
                    "Solver": s,
                    "Evaluated_Runs": cnt,
                    "Target_Runs": target_runs,
                    "Status": "Complete" if cnt >= target_runs else ("Partial" if cnt > 0 else "Missing")
                })

df_eval_audit = pd.DataFrame(eval_records)
total_cells = len(df_eval_audit)
completed_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] >= target_runs])
partial_cells = len(df_eval_audit[(df_eval_audit["Evaluated_Runs"] > 0) & (df_eval_audit["Evaluated_Runs"] < target_runs)])
missing_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0])
completion_pct = (completed_cells / total_cells) * 100.0 if total_cells > 0 else 0.0

# SQLite Evolutionary Phase Audit
if not loader.df_exp.empty:
    group_cols = [c for c in ["dim", "noise_std", "problem_id", "prompt_strategy"] if c in loader.df_exp.columns]
    db_summary = loader.df_exp.groupby(group_cols).size().reset_index(name="experiment_count")
    print(f"🧬 SQLite Evolutionary Experiments Logged: {len(loader.df_exp)} across {len(loader.df_iter)} iterations.")

print(f"🎯 Benchmark Evaluation Matrix (IOH Logs): {completed_cells}/{total_cells} cells completed ({completion_pct:.1f}% at N ≥ {target_runs} runs, {partial_cells} partial).")


🧬 SQLite Evolutionary Experiments Logged: 632 across 5264 iterations.
🎯 Benchmark Evaluation Matrix (IOH Logs): 472/600 cells completed (78.7% at N ≥ 20 runs, 0 partial).


In [4]:
# ── 4. Dynamic Markdown Audit Report Generator ───────────────────────────
missing_cells_df = df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0]
partial_cells_df = df_eval_audit[(df_eval_audit["Evaluated_Runs"] > 0) & (df_eval_audit["Evaluated_Runs"] < target_runs)]
target_cells_df  = df_eval_audit[df_eval_audit["Evaluated_Runs"] == target_runs]
over_cells_df    = df_eval_audit[df_eval_audit["Evaluated_Runs"] > target_runs]

n_total = len(df_eval_audit)
n_miss = len(missing_cells_df)
n_part = len(partial_cells_df)
n_target = len(target_cells_df)
n_over = len(over_cells_df)

report_lines = []
report_lines.append("# 📋 Experimental Matrix Coverage & Sample Imbalance Audit Report\n")
report_lines.append(f"**Generated:** `notebooks/04_audit.ipynb` | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n")
report_lines.append("### 📊 High-Level Status Breakdown")
report_lines.append(f"- **Total Experimental Cells Planned:** `{n_total}`")
report_lines.append(f"- 🔴 **Missing Conditions (0 Runs):** `{n_miss}` ({n_miss/n_total*100:.1f}%)")
report_lines.append(f"- 🟡 **Partial / Interrupted Runs (<{target_runs} Runs):** `{n_part}` ({n_part/n_total*100:.1f}%)")
report_lines.append(f"- 🟢 **Exact Target Met (N = {target_runs} Runs):** `{n_target}` ({n_target/n_total*100:.1f}%)")
report_lines.append(f"- 🔵 **Over-Sampled / Imbalanced (N > {target_runs} Runs):** `{n_over}` ({n_over/n_total*100:.1f}%)\n")
report_lines.append("---\n")

report_lines.append("## 1. Completion Rate by Dimension\n")
report_lines.append(f"| Dimension | Total Cells | Fully Completed (N ≥ {target_runs}) | Partial (<{target_runs}) | Missing (0) | Completion Rate |")
report_lines.append("|---|---|---|---|---|---|")
for d in loader.dims:
    sub = df_eval_audit[df_eval_audit["Dimension"] == f"{d}D"]
    tot = len(sub)
    full = len(sub[sub["Evaluated_Runs"] >= target_runs])
    part = len(sub[(sub["Evaluated_Runs"] > 0) & (sub["Evaluated_Runs"] < target_runs)])
    miss = len(sub[sub["Evaluated_Runs"] == 0])
    rate = (full / tot) * 100.0 if tot > 0 else 0.0
    report_lines.append(f"| **{d}D** | {tot} | {full} | {part} | {miss} | {rate:.1f}% |")
report_lines.append("\n---\n")

report_lines.append("## 2. Sample Size Imbalance by Solver\n")
report_lines.append("| Solver | Category | Evaluated Cells | Missing Cells | Mean Runs (N) | Min Runs | Max Runs |")
report_lines.append("|---|---|---|---|---|---|---|")
for s in loader.all_solvers:
    sub = df_eval_audit[df_eval_audit["Solver"] == s]
    tot = len(sub)
    comp = len(sub[sub["Evaluated_Runs"] >= target_runs])
    miss = len(sub[sub["Evaluated_Runs"] == 0])
    runs_s = sub[sub["Evaluated_Runs"] > 0]["Evaluated_Runs"]
    mean_n = runs_s.mean() if len(runs_s) > 0 else 0.0
    min_n = runs_s.min() if len(runs_s) > 0 else 0
    max_n = runs_s.max() if len(runs_s) > 0 else 0
    arch = s.split(" / ")[0] if " / " in s else "Classical Baseline"
    report_lines.append(f"| **{s}** | {arch} | {comp}/{tot} | {miss} | **{mean_n:.1f}** | {min_n} | {max_n} |")
report_lines.append("\n---\n")

report_lines.append("## 3. Actionable Checklist: Missing Experiments (0 Runs)\n")
if missing_cells_df.empty:
    report_lines.append("🎉 **No missing conditions! Full coverage achieved.**\n")
else:
    report_lines.append("| # | Dimension | Noise Regime | Problem Name | Problem Class | Solver | Action Required |")
    report_lines.append("|---|---|---|---|---|---|---|")
    for idx, (_, row) in enumerate(missing_cells_df.iterrows(), 1):
        report_lines.append(f"| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | {row["Hardness_Class"]} | **{row["Solver"]}** | 🔴 **Execute N={target_runs} Runs** |")
report_lines.append("\n---\n")

report_lines.append(f"## 4. Actionable Checklist: Partial / Interrupted Experiments (< {target_runs} Runs)\n")
if partial_cells_df.empty:
    report_lines.append("🎉 **No partial runs detected!**\n")
else:
    report_lines.append(f"| # | Dimension | Noise Regime | Problem Name | Solver | Completed Runs | Remaining to Target ({target_runs} - N) |")
    report_lines.append("|---|---|---|---|---|---|---|")
    for idx, (_, row) in enumerate(partial_cells_df.iterrows(), 1):
        rem = target_runs - int(row["Evaluated_Runs"])
        report_lines.append(f"| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | **{row["Solver"]}** | ⚠️ {row["Evaluated_Runs"]}/{target_runs} | 🟡 **Run +{rem} more** |")

report_path = AUDIT_DIR / "experimental_coverage_audit.md"
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
print(f"✅ Dynamic Audit Report written to: {report_path}")


✅ Dynamic Audit Report written to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/audit/experimental_coverage_audit.md


In [5]:
# ── 5. Render High-Contrast Status & Imbalance Matrix (Unified All Models & Baselines) ───
def render_unified_matrix(loader):
    llm_solvers = getattr(loader, "llm_solvers", [s for s in loader.all_solvers if not s.startswith("Baseline")])
    classical_solvers = getattr(loader, "classical_solvers", [s for s in loader.all_solvers if s.startswith("Baseline")])
    solvers = llm_solvers + classical_solvers
    
    n_rows = len(loader.dims)
    n_cols = len(loader.noise_levels)
    t_runs = getattr(service, "target_runs", 20)

    def get_status_code(cnt):
        if cnt == 0: return 0.0          # Missing (Red)
        elif cnt < t_runs: return 1.0    # Partial (Amber)
        elif cnt == t_runs: return 2.0   # Target Met (Green)
        else: return 3.0                 # Over-Sampled / Imbalance (Blue)

    def get_status_text(cnt):
        if cnt == 0: return "❌ 0"
        elif cnt < t_runs: return f"⚠️ {cnt}/{t_runs}"
        elif cnt == t_runs: return f"✅ {t_runs}"
        else: return f"🔷 {cnt}"

    colorscale = [
        [0.00, "#FFCDD2"], [0.24, "#FFCDD2"], # Missing (Soft Red)
        [0.26, "#FFE082"], [0.49, "#FFE082"], # Partial (Soft Amber)
        [0.51, "#C8E6C9"], [0.74, "#C8E6C9"], # Target Met (Soft Green)
        [0.76, "#BBDEFB"], [1.00, "#BBDEFB"]  # Over-Sampled / Imbalance (Soft Blue)
    ]

    subplot_titles = []
    coords_map = {}
    for r_idx, d in enumerate(loader.dims, 1):
        for c_idx, n in enumerate(loader.noise_levels, 1):
            label = "Clean (σ=0.0)" if n == 0.0 else f"Noisy (σ={n})"
            total = len(solvers) * len(loader.problem_ids)
            done = sum(1 for s in solvers for p in loader.problem_ids if loader.eval_counts[(d, n, p)][s] >= t_runs)
            pct = (done / total) * 100 if total > 0 else 0
            subplot_titles.append(f"<b>{d}D — {label}</b>  <span style=\"font-size:11px; color:#555;\">({done}/{total} done · {pct:.0f}%)</span>")
            coords_map[(d, n)] = (r_idx, c_idx)

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.10,
        vertical_spacing=0.08
    )

    prob_labels = [BBOBFunction.get_name(p) for p in loader.problem_ids]

    for d in loader.dims:
        for n in loader.noise_levels:
            r, c = coords_map[(d, n)]
            z_vals, text_vals = [], []
            for s in solvers:
                row_z, row_t = [], []
                for p in loader.problem_ids:
                    cnt = loader.eval_counts[(d, n, p)][s]
                    row_z.append(get_status_code(cnt))
                    row_t.append(get_status_text(cnt))
                z_vals.append(row_z)
                text_vals.append(row_t)
                
            fig.add_trace(
                go.Heatmap(
                    z=z_vals,
                    x=prob_labels,
                    y=solvers,
                    text=text_vals,
                    texttemplate="<b>%{text}</b>",
                    textfont=dict(size=10, family="Inter, Helvetica, Arial, sans-serif", color="#1E293B"),
                    colorscale=colorscale,
                    zmin=0.0, zmax=3.0,
                    showscale=False,
                    xgap=2, ygap=2
                ),
                row=r, col=c
            )
            if c == 1:
                fig.update_yaxes(autorange="reversed", row=r, col=c, tickfont=dict(size=10, family="Inter, sans-serif"))
            else:
                fig.update_yaxes(showticklabels=False, autorange="reversed", row=r, col=c)
            fig.update_xaxes(tickangle=-20, row=r, col=c, tickfont=dict(size=10, family="Inter, sans-serif"))

    fig.update_layout(
        template="plotly_white",
        title=dict(
            text="<b>Global Experimental Matrix Coverage & Sample Imbalance — All LLM Models & Baselines</b><br>" +
                 "<sup><b>Legend:</b>  " +
                 "<span style=\"color:#D32F2F;\">■</span> <b>❌ 0 (Missing)</b>   &nbsp;|&nbsp;   " +
                 f"<span style=\"color:#F57C00;\">■</span> <b>⚠️ &lt;{t_runs} (Partial)</b>   &nbsp;|&nbsp;   " +
                 f"<span style=\"color:#2E7D32;\">■</span> <b>✅ {t_runs} (Target Met)</b>   &nbsp;|&nbsp;   " +
                 f"<span style=\"color:#1976D2;\">■</span> <b>🔷 &gt;{t_runs} (Over-Sampled / Imbalance)</b></sup>",
            x=0.02, y=0.99,
            font=dict(size=13, color="#0F172A", family="Inter, Helvetica, Arial, sans-serif")
        ),
        width=1280, height=360 * n_rows + 60,
        margin=dict(l=220, r=40, t=110, b=50),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=10, color="#0F172A")
    )

    fig.show()
    return fig

fig = render_unified_matrix(loader)


In [6]:
# ── 6. Missing Experiment Action Plan & CLI Dispatcher ───────────────────
missing_df = df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0]
if not missing_df.empty:
    print(f"⚠️ {len(missing_df)} Experimental Conditions Require Execution:")
    for _, r in missing_df.iterrows():
        print(f"  • Dimension {r['Dimension']} | {r['Noise']} | {r['Problem_Name']} | Solver: {r['Solver']}")
else:
    print("🎉 All experimental matrix conditions are completely evaluated!")


⚠️ 128 Experimental Conditions Require Execution:
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: Qwen2.5-Coder-3B / baseline
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: Qwen2.5-Coder-3B / guided
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: Qwen2.5-Coder-3B / thinking
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: Qwen2.5-Coder-3B / baseline
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: Qwen2.5-Coder-3B / guided
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: Qwen2.5-Coder-3B / thinking
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: Qwen2.5-Coder-3B / vectorization
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: Qwen2.5-Coder-3B / baseline
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: Qwen2.5-Coder-3B / guided
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: Qwen2.5-Coder-3B / thinking
  • Dimension 2D | σ=0.0 | Rastrigin Multi-Modal (f15) | Solver: Qwen2.5-Coder-3B / baseline
  • Dimension 2D | σ=0.0 | Rastrigin Multi-Modal (f15) | Solver: Qwen